# 🎯 AI Interview Coach — Model Training (Google Colab)

This notebook fine-tunes 3 AI models for the Interview Coach:
1. **Question Generator** — FLAN-T5-small + LoRA (generates interview questions from resume)
2. **Answer Evaluator** — sentence-transformers (scores answer quality/relevance)
3. **Confidence Classifier** — PyTorch neural network (classifies speaking confidence)

**After training:** Download the `trained_models/` folder and place it in your project at `training/models_output/`

⚠️ Make sure to select **GPU runtime** (Runtime → Change runtime type → T4 GPU)

## Cell 1: Install Dependencies

In [ ]:
!pip install -q torch transformers datasets peft accelerate sentence-transformers scikit-learn tqdm

## Cell 2: Check GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

## Cell 3: Prepare Datasets

In [ ]:
import os
import json
import random
import numpy as np

os.makedirs("data", exist_ok=True)
os.makedirs("trained_models/question_generator/final", exist_ok=True)
os.makedirs("trained_models/answer_evaluator/final", exist_ok=True)
os.makedirs("trained_models/confidence_classifier", exist_ok=True)

# ═══════════════════════════════════════════════════
# Dataset 1: Question Generation
# ═══════════════════════════════════════════════════
print("Preparing Question Generation Dataset...")

qg_samples = []

# Seed interview question patterns
seed_patterns = [
    ("Skills: Python, Django, REST APIs", "Can you explain how Django's ORM handles database queries and what N+1 query problem is?"),
    ("Skills: Python, Flask, microservices", "How would you design a RESTful API using Flask? Walk me through your approach."),
    ("Skills: Python, pandas, data analysis", "How do you handle missing data in a pandas DataFrame? What strategies do you use?"),
    ("Skills: React, TypeScript, Next.js", "Explain the difference between useEffect and useLayoutEffect in React."),
    ("Skills: React, Redux, state management", "How do you manage global state in a large React application?"),
    ("Skills: Machine Learning, TensorFlow, NLP", "What is overfitting and how do you prevent it in a neural network?"),
    ("Skills: Deep Learning, PyTorch, Computer Vision", "Explain the architecture of a CNN and when you'd use it."),
    ("Skills: PostgreSQL, MongoDB, Redis", "When would you choose a NoSQL database over a relational one?"),
    ("Skills: Docker, Kubernetes, CI/CD", "Describe how you would set up a CI/CD pipeline from scratch."),
    ("Skills: AWS, Lambda, S3, DynamoDB", "How would you design a serverless architecture for a web application?"),
    ("Experience: 3 years, Team Lead", "Tell me about a time you had to make a difficult decision as a team lead."),
    ("Experience: 2 years, cross-functional teams", "Describe a situation where you disagreed with a colleague's approach."),
    ("Experience: Software Engineer, startup", "Tell me about a project that failed. What did you learn from it?"),
    ("Experience: 4 years, multiple deadlines", "How do you prioritize when you have multiple urgent tasks?"),
    ("Experience: Junior developer, 1 year", "Where do you see yourself growing technically in the next 2 years?"),
    ("Job Title: Software Engineer", "Describe your approach to writing clean, testable code."),
    ("Job Title: Data Scientist", "How do you validate whether your model generalizes well to unseen data?"),
    ("Job Title: Frontend Developer", "How do you ensure your web application is accessible to all users?"),
    ("Job Title: Backend Developer", "How do you handle database migrations in a production environment?"),
    ("Job Title: DevOps Engineer", "What monitoring and alerting strategies do you use for production systems?"),
    ("Skills: Flutter, Dart, mobile development", "How do you manage state in a Flutter application?"),
    ("Skills: Flutter, Firebase, REST APIs", "Describe how you handle offline data persistence in a mobile app."),
    ("Skills: Java, Spring Boot, Microservices", "Explain dependency injection and how Spring Boot implements it."),
    ("Skills: Node.js, Express, MongoDB", "How do you handle authentication and authorization in a Node.js API?"),
    ("Skills: C++, algorithms, data structures", "Explain the time complexity of common sorting algorithms."),
    ("Skills: Swift, iOS, UIKit", "What is the difference between value types and reference types in Swift?"),
    ("Skills: Go, gRPC, microservices", "How does Go handle concurrency and what are goroutines?"),
    ("Skills: Rust, systems programming", "Explain Rust's ownership model and how it prevents memory errors."),
    ("Skills: PHP, Laravel, Vue.js", "How do you implement middleware in a Laravel application?"),
    ("Skills: Kotlin, Android, Jetpack Compose", "How does Jetpack Compose differ from traditional XML layouts?"),
]

for context, question in seed_patterns:
    qg_samples.append({"context": context, "question": question})

# Augment with skill combinations
skill_sets = [
    ["Python", "Django", "PostgreSQL"],
    ["JavaScript", "React", "Node.js"],
    ["Java", "Spring Boot", "MySQL"],
    ["Python", "TensorFlow", "scikit-learn"],
    ["Go", "Docker", "Kubernetes"],
    ["TypeScript", "Angular", "MongoDB"],
    ["C++", "algorithms", "data structures"],
    ["Python", "FastAPI", "Redis"],
    ["Flutter", "Dart", "Firebase"],
    ["Kotlin", "Android", "Room"],
    ["Python", "pandas", "NumPy", "matplotlib"],
    ["React Native", "Firebase", "Redux"],
    ["Rust", "systems programming", "WebAssembly"],
    ["PHP", "Laravel", "Vue.js"],
    ["Ruby", "Rails", "Sidekiq"],
    ["Swift", "iOS", "Core Data"],
    ["Scala", "Spark", "Kafka"],
    ["C#", ".NET", "Azure"],
]

question_templates = [
    "Explain your experience with {skill}. What projects have you built?",
    "What are the most important concepts in {skill} that a developer should master?",
    "How would you debug a performance issue involving {skill}?",
    "Describe a challenging problem you solved using {skill}.",
    "How do you keep up with updates and best practices in {skill}?",
    "What are the trade-offs when using {skill} versus its alternatives?",
    "How has {skill} evolved in recent years and what trends do you see?",
    "Walk me through how {skill} works under the hood.",
    "Can you give an example of a real-world application of {skill}?",
    "What common mistakes do beginners make with {skill}?",
]

for skills in skill_sets:
    skill_str = f"Skills: {', '.join(skills)}"
    for skill in skills[:3]:
        for template in random.sample(question_templates, 4):
            q = template.format(skill=skill)
            qg_samples.append({"context": skill_str, "question": q})

# Add SQuAD data for general question patterns
try:
    from datasets import load_dataset
    squad = load_dataset("squad", split="train[:3000]")
    for item in squad:
        qg_samples.append({"context": item["context"][:200], "question": item["question"]})
    print(f"  Added {len(squad)} SQuAD samples")
except:
    print("  Could not load SQuAD, continuing with custom data")

random.shuffle(qg_samples)
with open("data/qg_dataset.json", "w") as f:
    json.dump(qg_samples, f, indent=2)
print(f"  Question Generation: {len(qg_samples)} samples saved")


# ═══════════════════════════════════════════════════
# Dataset 2: Answer Evaluation
# ═══════════════════════════════════════════════════
print("\nPreparing Answer Evaluation Dataset...")

ae_samples = []

good_answers = [
    {"question": "Tell me about yourself.", "answer": "I'm a software engineer with 3 years of experience specializing in full-stack development. I graduated from FAST University with a degree in Computer Science. In my current role at TechCorp, I've led the development of two major products using React and Python. I'm passionate about building scalable systems and mentoring junior developers.", "keywords": ["experience", "education", "skills", "passion", "career"], "quality_score": 4.5},
    {"question": "Explain your experience with Python.", "answer": "I've been working with Python for 4 years now. I started with data analysis using pandas and NumPy, then moved to web development with Django and FastAPI. In my last project, I built a microservices architecture handling 10,000 requests per second. I also contributed to open-source Python libraries and regularly use pytest for testing.", "keywords": ["python", "project", "experience", "developed", "application"], "quality_score": 4.8},
    {"question": "How do you handle tight deadlines?", "answer": "I approach tight deadlines by first breaking down the work into priority tiers. I communicate early with stakeholders about what's achievable and what might need to be deferred. For example, last quarter we had a critical launch — I organized daily standups, delegated effectively, and we shipped on time by focusing on core features first.", "keywords": ["deadline", "prioritize", "communicate", "deliver", "manage"], "quality_score": 4.5},
    {"question": "What is REST API and how do you design one?", "answer": "REST stands for Representational State Transfer. When designing a REST API, I follow these principles: use proper HTTP methods like GET for reading, POST for creating, PUT for updating, and DELETE for removing resources. I use meaningful status codes, implement pagination for large datasets, version my APIs, and always include proper error handling with descriptive messages.", "keywords": ["rest", "api", "endpoint", "http", "json", "status"], "quality_score": 4.7},
    {"question": "Describe a challenging project.", "answer": "Last year, I led the migration of our monolithic application to microservices. The biggest challenge was maintaining data consistency across services. I implemented an event-driven architecture using RabbitMQ, created a comprehensive testing strategy, and we migrated incrementally over 3 months. The result was a 60% improvement in deployment speed.", "keywords": ["challenge", "problem", "solution", "result", "approach"], "quality_score": 4.9},
    {"question": "Explain state management in Flutter.", "answer": "In Flutter, I primarily use Provider for state management because it's well-integrated with the widget tree. For more complex applications, I've used BLoC pattern which separates business logic from UI. I structure my app with ChangeNotifier classes for state and use Consumer widgets to rebuild only the parts of the UI that depend on specific state changes.", "keywords": ["flutter", "state", "provider", "bloc", "widget"], "quality_score": 4.6},
]

medium_answers = [
    {"question": "Tell me about yourself.", "answer": "I'm a developer. I know Python and JavaScript. I've worked at a couple of companies. I like coding and learning new things.", "keywords": ["experience", "education", "skills", "passion", "career"], "quality_score": 2.5},
    {"question": "Explain your experience with React.", "answer": "I've used React for about a year. I know hooks and components. I've built a few projects with it. It's a good library for frontend.", "keywords": ["react", "project", "hooks", "components", "experience"], "quality_score": 2.3},
    {"question": "What are your strengths?", "answer": "I'm good at problem solving and I work hard. I learn quickly and I'm a team player.", "keywords": ["strength", "skill", "teamwork", "communication", "problem"], "quality_score": 2.0},
    {"question": "How do you handle conflicts in a team?", "answer": "I try to talk to people and understand their perspective. Usually we can find a compromise. Communication is important.", "keywords": ["communication", "listen", "understand", "compromise", "resolve"], "quality_score": 2.8},
]

poor_answers = [
    {"question": "Tell me about yourself.", "answer": "Um, I don't know, I just like computers I guess.", "keywords": ["experience", "education", "skills", "passion", "career"], "quality_score": 0.8},
    {"question": "Explain your experience with Python.", "answer": "Yeah, I've used Python. It's a programming language. I wrote some code with it.", "keywords": ["python", "project", "experience", "developed", "application"], "quality_score": 0.7},
    {"question": "Describe a challenging project.", "answer": "Uh, I can't really think of one right now. Maybe there was something but I don't remember.", "keywords": ["challenge", "problem", "solution", "result", "approach"], "quality_score": 0.5},
    {"question": "What is machine learning?", "answer": "It's like when computers learn stuff. I think it uses data or something.", "keywords": ["supervised", "unsupervised", "algorithm", "data", "train", "model"], "quality_score": 0.6},
]

ae_samples.extend(good_answers)
ae_samples.extend(medium_answers)
ae_samples.extend(poor_answers)

# Augment
augment_topics = [
    ("Docker", "3", "container orchestration, multi-stage builds"),
    ("AWS", "2", "Lambda, auto-scaling, CloudFormation"),
    ("machine learning", "4", "feature engineering, model deployment, A/B testing"),
    ("database design", "3", "normalization, indexing strategies, query optimization"),
    ("agile methodology", "2", "sprint planning, retrospectives, continuous delivery"),
    ("Flutter", "3", "custom widgets, platform channels, state management"),
    ("GraphQL", "2", "schema design, resolvers, subscriptions"),
    ("testing", "4", "unit tests, integration tests, TDD, mocking"),
]

for topic, years, advanced in augment_topics:
    q = f"What is your experience with {topic}?"
    kw = [topic.lower(), "experience", "project", "build"]
    # Good
    ae_samples.append({"question": q, "answer": f"I have {years} years of experience with {topic}. In my most recent project, I used {topic} to build a scalable solution that handled thousands of users. I'm comfortable with both the fundamentals and advanced concepts like {advanced}.", "keywords": kw, "quality_score": 4.2 + random.uniform(-0.3, 0.3)})
    # Poor
    ae_samples.append({"question": q, "answer": f"I know {topic} a bit. I've used it sometimes.", "keywords": kw, "quality_score": 1.0 + random.uniform(-0.2, 0.3)})

random.shuffle(ae_samples)
with open("data/ae_dataset.json", "w") as f:
    json.dump(ae_samples, f, indent=2)
print(f"  Answer Evaluation: {len(ae_samples)} samples saved")


# ═══════════════════════════════════════════════════
# Dataset 3: Confidence Classification
# ═══════════════════════════════════════════════════
print("\nPreparing Confidence Classification Dataset...")

conf_samples = []

# Features: [wpm, pitch_cv, pause_ratio, volume_cv, pauses_per_min]
# Labels: 0=nervous, 1=moderate, 2=confident

for _ in range(500):
    conf_samples.append({"features": [np.random.uniform(120, 160), np.random.uniform(0.08, 0.25), np.random.uniform(0.05, 0.15), np.random.uniform(0.15, 0.30), np.random.uniform(1, 4)], "label": 2})

for _ in range(500):
    conf_samples.append({"features": [np.random.uniform(100, 180), np.random.uniform(0.05, 0.15), np.random.uniform(0.15, 0.25), np.random.uniform(0.30, 0.50), np.random.uniform(4, 8)], "label": 1})

for _ in range(500):
    wpm = np.random.choice([np.random.uniform(60, 100), np.random.uniform(180, 240)])
    conf_samples.append({"features": [wpm, np.random.uniform(0.02, 0.06), np.random.uniform(0.25, 0.45), np.random.uniform(0.50, 0.80), np.random.uniform(8, 15)], "label": 0})

random.shuffle(conf_samples)
with open("data/conf_dataset.json", "w") as f:
    json.dump(conf_samples, f, indent=2)
print(f"  Confidence Classification: {len(conf_samples)} samples saved")

print("\n✅ All datasets prepared!")

## Cell 4: Train Question Generator (FLAN-T5 + LoRA)

Fine-tunes `google/flan-t5-small` (80M params) with LoRA for efficient training on GPU.

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType

print("="*60)
print("  Training Question Generator (FLAN-T5 + LoRA)")
print("="*60)

# Load data
with open("data/qg_dataset.json") as f:
    raw_data = json.load(f)

formatted = []
for item in raw_data:
    formatted.append({
        "input_text": f"Generate an interview question for: {item['context']}",
        "output_text": item["question"],
    })

# Load model
BASE_MODEL = "google/flan-t5-small"
print(f"\nLoading {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q", "v"], bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize
def tokenize(examples):
    inputs = tokenizer(examples["input_text"], max_length=256, truncation=True, padding="max_length")
    outputs = tokenizer(examples["output_text"], max_length=128, truncation=True, padding="max_length")
    inputs["labels"] = outputs["input_ids"]
    return inputs

split = int(len(formatted) * 0.9)
train_ds = Dataset.from_list(formatted[:split]).map(tokenize, batched=True, remove_columns=["input_text", "output_text"])
val_ds = Dataset.from_list(formatted[split:]).map(tokenize, batched=True, remove_columns=["input_text", "output_text"])

print(f"  Train: {len(train_ds)} | Val: {len(val_ds)}")

# Train
args = Seq2SeqTrainingArguments(
    output_dir="./qg_checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
    tokenizer=tokenizer,
)

print("\n🚀 Training...")
result = trainer.train()
print(f"  Final loss: {result.training_loss:.4f}")

eval_result = trainer.evaluate()
print(f"  Eval loss: {eval_result['eval_loss']:.4f}")

# Save
out = "trained_models/question_generator/final"
trainer.save_model(out)
tokenizer.save_pretrained(out)

# Save metrics
with open(f"{out}/training_metrics.json", "w") as f:
    json.dump({"training_loss": result.training_loss, "eval_loss": eval_result["eval_loss"], "epochs": 5}, f, indent=2)

# Test
print("\n🧪 Test generation:")
for test in ["Skills: Python, Django, PostgreSQL", "Job Title: Data Scientist", "Skills: Flutter, Dart, Firebase"]:
    inputs = tokenizer(f"Generate an interview question for: {test}", return_tensors="pt", max_length=256, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=128, num_beams=3, do_sample=True, temperature=0.8)
    print(f"  [{test}] → {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

print("\n✅ Question Generator trained!")

## Cell 5: Train Answer Evaluator (sentence-transformers)

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

print("="*60)
print("  Training Answer Evaluator (sentence-transformers)")
print("="*60)

# Load data
with open("data/ae_dataset.json") as f:
    ae_data = json.load(f)

# Create training pairs
examples = []
for item in ae_data:
    similarity = item["quality_score"] / 5.0
    q_context = item["question"] + " " + " ".join(item.get("keywords", [])[:5])
    examples.append(InputExample(texts=[q_context, item["answer"]], label=similarity))

    # Add negative examples for poor answers
    if item["quality_score"] < 2.0:
        others = [d["question"] for d in ae_data if d["question"] != item["question"]]
        if others:
            examples.append(InputExample(texts=[random.choice(others), item["answer"]], label=0.1))

random.shuffle(examples)
split = int(len(examples) * 0.85)
train_examples = examples[:split]
val_examples = examples[split:]

print(f"  Train pairs: {len(train_examples)} | Val pairs: {len(val_examples)}")

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Train
train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model=model)

val_s1 = [e.texts[0] for e in val_examples]
val_s2 = [e.texts[1] for e in val_examples]
val_scores = [e.label for e in val_examples]
evaluator = EmbeddingSimilarityEvaluator(val_s1, val_s2, val_scores, name="interview-val")

out_path = "trained_models/answer_evaluator/final"
os.makedirs(out_path, exist_ok=True)

print("\n🚀 Training...")
model.fit(
    train_objectives=[(train_loader, train_loss)],
    evaluator=evaluator,
    epochs=10,
    warmup_steps=int(len(train_loader) * 0.1),
    optimizer_params={"lr": 2e-5},
    output_path=out_path,
    evaluation_steps=max(1, len(train_loader) // 5),
    save_best_model=True,
    show_progress_bar=True,
)

# Test
print("\n🧪 Test similarity:")
test_model = SentenceTransformer(out_path)
pairs = [
    ("Explain Python experience project developed", "I have 4 years of Python experience. I built web apps with Django."),
    ("Explain Python experience project developed", "I like pizza and movies."),
    ("How do you handle deadlines prioritize", "I break work into priorities and communicate with stakeholders."),
    ("How do you handle deadlines prioritize", "I don't know, I just work harder maybe."),
]
for q, a in pairs:
    emb = test_model.encode([q, a])
    sim = np.dot(emb[0], emb[1]) / (np.linalg.norm(emb[0]) * np.linalg.norm(emb[1]))
    print(f"  Sim={sim:.3f} | Q: {q[:40]}... | A: {a[:40]}...")

# Save metrics
with open(f"{out_path}/training_metrics.json", "w") as f:
    json.dump({"base_model": "all-MiniLM-L6-v2", "epochs": 10, "pairs": len(train_examples)}, f, indent=2)

print("\n✅ Answer Evaluator trained!")

## Cell 6: Train Confidence Classifier (PyTorch Neural Network)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

print("="*60)
print("  Training Confidence Classifier (PyTorch NN)")
print("="*60)

# Load data
with open("data/conf_dataset.json") as f:
    conf_data = json.load(f)

features = np.array([d["features"] for d in conf_data], dtype=np.float32)
labels = np.array([d["label"] for d in conf_data], dtype=np.int64)

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42, stratify=labels)

# Normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"  Train: {len(X_train)} | Test: {len(X_test)}")

# Model
class ConfidenceNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 3),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConfidenceNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.LongTensor(y_test).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)

# Train
print("\n🚀 Training for 80 epochs...")
best_acc = 0
for epoch in range(80):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == by).sum().item()
        total += by.size(0)
    scheduler.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_out = model(X_test_t)
            test_acc = (test_out.argmax(1) == y_test_t).float().mean().item() * 100
        if test_acc > best_acc:
            best_acc = test_acc
        print(f"  Epoch {epoch+1:3d}/80 | Loss: {total_loss/len(train_loader):.4f} | Train: {100*correct/total:.1f}% | Test: {test_acc:.1f}%")

# Final eval
model.eval()
with torch.no_grad():
    preds = model(X_test_t).argmax(1).cpu().numpy()
print(f"\n📊 Classification Report:")
print(classification_report(y_test, preds, target_names=["nervous", "moderate", "confident"]))

# Save
out_dir = "trained_models/confidence_classifier"
torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "input_dim": 5, "num_classes": 3,
}, f"{out_dir}/confidence_net.pth")

with open(f"{out_dir}/scaler_params.json", "w") as f:
    json.dump({"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()}, f, indent=2)

with open(f"{out_dir}/training_metrics.json", "w") as f:
    json.dump({"best_accuracy": best_acc, "epochs": 80, "classes": ["nervous", "moderate", "confident"]}, f, indent=2)

print(f"\n✅ Confidence Classifier trained! Best accuracy: {best_acc:.1f}%")

## Cell 7: Download Trained Models

This will zip all trained models so you can download them and place in your project at `training/models_output/`

In [ ]:
!zip -r trained_models.zip trained_models/

from google.colab import files
files.download("trained_models.zip")

print("\n" + "="*60)
print("  ✅ ALL MODELS TRAINED & DOWNLOADED!")
print("="*60)
print()
print("  📁 Place the downloaded files in your project:")
print("  Extract trained_models.zip into:")
print("  your-project/training/models_output/")
print()
print("  The folder structure should be:")
print("  training/models_output/")
print("  ├── question_generator/final/  (FLAN-T5 LoRA model)")
print("  ├── answer_evaluator/final/    (sentence-transformers)")
print("  └── confidence_classifier/     (PyTorch NN)")
print()
print("  Then run 'start.bat' to launch the website!")
print("="*60)